In [1]:
import logging
import pathlib
from argparse import ArgumentParser, RawTextHelpFormatter
from dataclasses import dataclass
from functools import partial
from typing import Callable

import torch
import torchaudio
import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/pytorch_audio/audio/examples/asr/emformer_rnnt/')
from common import MODEL_TYPE_LIBRISPEECH
from torchaudio.pipelines import EMFORMER_RNNT_BASE_LIBRISPEECH
from torchaudio.pipelines import RNNTBundle
import torchaudio.transforms as T

sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset
from src.text import load_text_encoder

import fuzzy 
from tqdm.notebook import tqdm
import pandas as pd

### Pytorch demo format 

In [2]:
@dataclass
class Config:
    dataset: Callable
    bundle: RNNTBundle

In [3]:

_CONFIGS = {
    MODEL_TYPE_LIBRISPEECH: Config(
        partial(torchaudio.datasets.LIBRISPEECH, url="test-clean"),
        EMFORMER_RNNT_BASE_LIBRISPEECH)
}

### Set params 

In [4]:
model_type = 'librispeech'

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set

mode = 'word'                       # 'character'/'word'/'subword'
vocab_file = '/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/tests/sample_data/gigaspeech_words-10k.vocab'


### Load model

In [8]:
# dataset 
clean_dataset = Dataset(path, dev_split, None, 1)

# model 
bundle = _CONFIGS[model_type].bundle
decoder = bundle.get_decoder().cuda()
token_processor = bundle.get_token_processor()
feature_extractor = bundle.get_feature_extractor().cuda()
streaming_feature_extractor = bundle.get_streaming_feature_extractor().cuda()
hop_length = bundle.hop_length
num_samples_segment = bundle.segment_length * hop_length
num_samples_segment_right_context = num_samples_segment + bundle.right_context_length * hop_length

Using custom data configuration default-6e977ecf05190e74
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


#### Single words recorded at 44.1kHz - resample to 16kHz on the fly

In [6]:
# resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
#                    rolloff=0.9475937167399596, 
#                    resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
#     wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


Unpack huggingface dataset from custom dataset object for convenience 

In [9]:
updated_dataset = clean_dataset.dataset.map(load_audio)

Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c28c661dad041df8.arrow


#### Run Single-Word evaluation

In [33]:
out_file = '../result/emformer_single_word.csv'

results = []

# with open(out_file,'w',encoding='UTF-8') as f:
#                 f.write('idx\tbeam\thyp\ttruth\n')

for ix, sample in enumerate( tqdm(updated_dataset)):
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()
#     Streaming decode.
    state, hypothesis, stream_transcript = None, None, ''
#     print("guess:")
    for idx in range(0, len(waveform), num_samples_segment):
        segment = waveform[idx : idx + num_samples_segment_right_context]
        segment = torch.nn.functional.pad(segment, (0, num_samples_segment_right_context - len(segment)))
        with torch.no_grad():
            features, length = streaming_feature_extractor(segment)
            hypos, state = decoder.infer(features, length, 1, state=state, hypothesis=hypothesis)
        hypothesis = hypos[0]
        transcript = token_processor(hypothesis.tokens, lstrip=False)
        stream_transcript += transcript.strip(' ')
#         print(transcript, end="", flush=True)

    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 1)
    transcript = token_processor(hypos[0].tokens)
    
    results.append({'hyp': transcript, 
                    'stream hyp': stream_transcript, 
                    'truth': truth})
#     print([token_processor(hypos[ix].tokens) for ix in range(10)])
#     print(f"truth: {sample[1]}")
#     print()

#     hypos = [token_processor(hypos[ix].tokens) for ix in range(10)]
#     with open(out_file, 'a', encoding='UTF-8') as f:
#         for b, hyp in enumerate(hypos):
#             f.write('\t'.join([truth, str(b), hyp, truth])+'\n')


  0%|          | 0/200 [00:00<?, ?it/s]

In [34]:
results = pd.DataFrame(results)

In [35]:
results.head(5)

,hyp,stream hyp,truth
0,the,,the
1,and,and,and
2,two,two,to
3,of,of,of
4,a,a,a


In [36]:
# top_match_guesses = results[(results['hyp'] == results['truth']) & (results['beam']==0)]
top_match_guesses = results[(results['hyp'] == results['truth'])]

percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

166 matched guesses; 83.0% correct


,hyp,stream hyp,truth
0,the,,the
1,and,and,and
3,of,of,of
4,a,a,a
5,that,,that


In [38]:
results['stream hyp'] = results['stream hyp'].str.strip(' ')

In [39]:
# top_match_guesses = results[(results['hyp'] == results['truth']) & (results['beam']==0)]
top_match_guesses = results[(results['stream hyp'] == results['truth'])]

percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

108 matched guesses; 54.0% correct


,hyp,stream hyp,truth
1,and,and,and
3,of,of,of
4,a,a,a
8,you,you,you
9,is,is,is


In [42]:

dmeta = fuzzy.DMetaphone()

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['stram_hyp_dmeta'] = results['stream hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)

results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

In [43]:
# top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta']) & (results['beam']==0)]
top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta'])]

percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

182 matched guesses; 91.0% correct


,hyp,stream hyp,truth,hyp_dmeta,truth_dmeta,stram_hyp_dmeta
0,the,,the,"[b'0', b'T']","[b'0', b'T']","[None, None]"
1,and,and,and,"[b'ANT', None]","[b'ANT', None]","[b'ANT', None]"
2,two,two,to,"[b'T', None]","[b'T', None]","[b'T', None]"
3,of,of,of,"[b'AF', None]","[b'AF', None]","[b'AF', None]"
4,a,a,a,"[b'A', None]","[b'A', None]","[b'A', None]"


In [44]:
# top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta']) & (results['beam']==0)]
top_match_guesses = results[(results['stram_hyp_dmeta'] == results['truth_dmeta'])]

percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

116 matched guesses; 58.0% correct


,hyp,stream hyp,truth,hyp_dmeta,truth_dmeta,stram_hyp_dmeta
1,and,and,and,"[b'ANT', None]","[b'ANT', None]","[b'ANT', None]"
2,two,two,to,"[b'T', None]","[b'T', None]","[b'T', None]"
3,of,of,of,"[b'AF', None]","[b'AF', None]","[b'AF', None]"
4,a,a,a,"[b'A', None]","[b'A', None]","[b'A', None]"
8,you,you,you,"[b'A', None]","[b'A', None]","[b'A', None]"


In [56]:
clean_resuts = results.drop(columns=['stream hyp', 'stram_hyp_dmeta'])

In [58]:
clean_resuts['condition'] = 'clean'

## On noise 0 dBSNR

In [45]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_0SNR']  # Name of data splits to be used as validation set

dataset = Dataset(path, dev_split, None, 1)

updated_dataset = dataset.dataset.map(load_audio)

Using custom data configuration default-c9dd0aeaa110b3ae
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-1aa1c950d7d80a14.arrow


In [48]:
# out_file = '../result/emformer_single_word_0dB_SNR.csv'

# with open(out_file,'w',encoding='UTF-8') as f:
#                 f.write('idx\tbeam\thyp\ttruth\n')



low_results = []

for sample in tqdm(updated_dataset):
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()
    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 1)
        
    transcript = token_processor(hypos[0].tokens)


    low_results.append({'hyp': transcript, 
                    'truth': truth})



  0%|          | 0/200 [00:00<?, ?it/s]

In [53]:
low_results = pd.DataFrame(low_results)
low_results['condition'] = '0dB SNR'

In [54]:

top_match_guesses = low_results[(low_results['hyp'] == low_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

45 matched guesses; 22.5% correct


,hyp,truth,condition
13,so,so,0dB SNR
27,like,like,0dB SNR
28,what,what,0dB SNR
32,about,about,0dB SNR
33,one,one,0dB SNR


In [60]:
dmeta = fuzzy.DMetaphone()

low_results['hyp_dmeta'] = low_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
low_results['truth_dmeta'] = low_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = low_results[(low_results['hyp_dmeta'] == low_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

49 matched guesses; 24.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
7,ay,i,0dB SNR,"[b'A', None]","[b'A', None]"
13,so,so,0dB SNR,"[b'S', None]","[b'S', None]"
27,like,like,0dB SNR,"[b'LK', None]","[b'LK', None]"
28,what,what,0dB SNR,"[b'AT', None]","[b'AT', None]"
29,now,know,0dB SNR,"[b'N', b'NF']","[b'N', b'NF']"


## On noise -5 dBSNR

In [61]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_-5SNR']  # Name of data splits to be used as validation set

dataset = Dataset(path, dev_split, None, 1)

updated_dataset = dataset.dataset.map(load_audio)


Using custom data configuration default-5683877c4c4495f8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c3c5b179e5247f84.arrow


In [62]:
# out_file = '../result/emformer_single_word_0dB_SNR.csv'

# with open(out_file,'w',encoding='UTF-8') as f:
#                 f.write('idx\tbeam\thyp\ttruth\n')



high_results = []

for sample in tqdm(updated_dataset):
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()
    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 1)
        
    transcript = token_processor(hypos[0].tokens)


    high_results.append({'hyp': transcript, 
                    'truth': truth})



  0%|          | 0/200 [00:00<?, ?it/s]

In [63]:
high_results = pd.DataFrame(high_results)
high_results['condition'] = '-5dB SNR'

In [64]:

top_match_guesses = high_results[(high_results['hyp'] == high_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(high_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

1 matched guesses; 0.5% correct


,hyp,truth,condition
87,see,see,-5dB SNR


In [66]:
dmeta = fuzzy.DMetaphone()

high_results['hyp_dmeta'] = high_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
high_results['truth_dmeta'] = high_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = high_results[(high_results['hyp_dmeta'] == high_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(high_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

1 matched guesses; 0.5% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
87,see,see,-5dB SNR,"[b'S', None]","[b'S', None]"


In [67]:
all_results = pd.concat([clean_resuts, low_results, high_results])

In [68]:
all_results

,hyp,truth,hyp_dmeta,truth_dmeta,condition
0,the,the,"[b'0', b'T']","[b'0', b'T']",clean
1,and,and,"[b'ANT', None]","[b'ANT', None]",clean
2,two,to,"[b'T', None]","[b'T', None]",clean
3,of,of,"[b'AF', None]","[b'AF', None]",clean
4,a,a,"[b'A', None]","[b'A', None]",clean
...,...,...,...,...,...
195,,twenty,"[None, None]","[b'TNT', None]",-5dB SNR
196,,each,"[None, None]","[b'AK', None]",-5dB SNR
197,,sure,"[None, None]","[b'SR', None]",-5dB SNR
198,,let,"[None, None]","[b'LT', None]",-5dB SNR


In [69]:
# all_results.to_pickle('../single_word_results/emformer_rnnt_single_word.pdpkl')